In [ ]:
# 1️⃣ Install dependencies
!pip install torch torchvision torchaudio
!pip install sentence-transformers
!pip install google-colab

In [ ]:
# 2️⃣ Imports
import json
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from google.colab import drive

In [ ]:
# 3️⃣ Mount Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 4️⃣ Load KB dataset
path = "/content/drive/MyDrive/Research_My_Component/miriad_cardiology.json"

with open(path, "r") as f:
    raw_data = json.load(f)

# Take first 1000 questions for training
kb_questions = [row["question"] for row in raw_data[:1000] if "question" in row]
print(f"Loaded {len(kb_questions)} questions for training")

Loaded 1000 questions for training


In [ ]:
# 5️⃣ Projection Layer
class ProjectionLayer(nn.Module):
    def __init__(self, in_dim=1024, out_dim=768):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)

projector = ProjectionLayer().cuda()

In [ ]:
# 6️⃣ Dataset Class
class ProjectionDataset(Dataset):
    def __init__(self, questions, inbedder_model, instructor_model):
        self.questions = questions
        self.inbedder = inbedder_model
        self.instructor = instructor_model

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        text = self.questions[idx]

        # Embed without autograd
        with torch.no_grad():
            inb_emb = self.inbedder.encode(text, convert_to_tensor=True, normalize_embeddings=True).detach()
            inst_emb = self.instructor.encode(f"Represent the cardiology question:\n{text}", convert_to_tensor=True, normalize_embeddings=True).detach()

        return inb_emb.float(), inst_emb.float()

In [ ]:
# 7️⃣ Load embedding models
inbedder_model = SentenceTransformer("KomeijiForce/inbedder-roberta-large")
instructor_model = SentenceTransformer("hkunlp/instructor-large")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at KomeijiForce/inbedder-roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 8️⃣ Prepare dataset + dataloader
dataset = ProjectionDataset(kb_questions, inbedder_model, instructor_model)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
# 9️⃣ Training loop
optimizer = torch.optim.Adam(projector.parameters(), lr=1e-4)
loss_fn = nn.CosineEmbeddingLoss()  # projected inbedder ~ instructor

for epoch in range(5):
    epoch_loss = 0
    for inb_emb, inst_emb in loader:
        inb_emb = inb_emb.cuda()
        inst_emb = inst_emb.cuda()

        projected = projector(inb_emb)
        target = torch.ones(inb_emb.size(0)).cuda()  # +1 similarity

        loss = loss_fn(projected, inst_emb, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Avg Loss: {epoch_loss/len(loader):.4f}")

Epoch 1, Avg Loss: 0.5086
Epoch 2, Avg Loss: 0.1168
Epoch 3, Avg Loss: 0.0698
Epoch 4, Avg Loss: 0.0636
Epoch 5, Avg Loss: 0.0606


In [ ]:
# 10️⃣ Save trained projector
torch.save(projector.state_dict(), "/content/drive/MyDrive/Research_My_Component/Projection_Layer/projector.pt")
print("✅ Projection layer trained and saved")

✅ Projection layer trained and saved
